# NBA production run — pinned warehouse

This is the NBA notebook; do not substitute `kaggle_mlb_run.ipynb`. It is orchestration only. It runs the standalone `nba-backend` pipeline against **wyattowalsh/basketball version 238**, writes the normalized cache outside the repository, and prepares only `nba-backend/data_delivery` for human-reviewed publication. It never commits or pushes.

In [ ]:
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

DATASET = "wyattowalsh/basketball"
DATASET_VERSION = "238"
EXPECTED_REPO = "andrewkemmer/sports_prediction_model"
REPO = Path("/kaggle/working/sports_prediction_model")
DELIVERY = Path("nba-backend/data_delivery")

os.environ["NBA_KAGGLE_DATASET_VERSION"] = DATASET_VERSION
os.environ["NBA_PUSH"] = "0"
# Keep the derived warehouse outside the checkout. Kaggle's working
# directory is ephemeral, but this also makes the scope explicit.
os.environ.setdefault("NBA_CACHE_DIR", "/kaggle/working/nba-cache")

if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{EXPECTED_REPO}.git", str(REPO)], check=True)
assert (REPO / "nba-backend/backend/master_pipeline.py").is_file()
print(f"cloned {EXPECTED_REPO} at NBA dataset pin {DATASET_VERSION}")

In [ ]:
subprocess.run(["pip", "install", "-q", "-r", "nba-backend/backend/requirements.txt"], cwd=REPO, check=True)
print("NBA dependencies installed")

In [ ]:
# Resolve the attached Kaggle input when the dataset is mounted by the
# notebook UI, otherwise download the exact pinned version. The resolver
# understands Kaggle's generated slug directories and nested export roots.
sys.path.insert(0, str(REPO / "nba-backend" / "backend"))
import ingestion

def find_warehouse(roots=None):
    search_roots = roots or [Path("/kaggle/input"), Path("/kaggle/working/nba-warehouse")]
    return ingestion.discover_warehouse(search_roots)

input_root = Path("/kaggle/input")
attached = sorted(path.name for path in input_root.iterdir()) if input_root.exists() else []
print(f"NBA attached inputs: {attached}")
target = Path("/kaggle/working/nba-warehouse")
source = find_warehouse()
if source is None:
    if target.exists():
        shutil.rmtree(target)
    try:
        subprocess.run([
            "kaggle", "datasets", "download", "-d", DATASET,
            "-v", DATASET_VERSION, "--unzip", "-p", str(target),
        ], check=True)
    except (FileNotFoundError, subprocess.CalledProcessError) as exc:
        raise RuntimeError(
            "NBA warehouse v238 is not mounted under /kaggle/input and the "
            "pinned Kaggle download failed. Attach wyattowalsh/basketball "
            "version 238 in Notebook Settings, then rerun."
        ) from exc
    for archive in target.rglob("*.zip"):
        with zipfile.ZipFile(archive) as bundle:
            bundle.extractall(target)
    source = find_warehouse([target])
if source is None or not source.exists():
    searched = [str(path) for path in (Path("/kaggle/input"), target)]
    raise RuntimeError(
        "No usable NBA warehouse found after resolving the pinned Kaggle "
        f"dataset. Searched: {searched}. Attach version 238 or check the "
        "download output for nba.duckdb, nba.sqlite, parquet/, or csv/."
    )
os.environ["NBA_KAGGLE_DATASET_PATH"] = str(source)
print(f"warehouse source: {source}")

In [ ]:
cmd = ["python", "nba-backend/backend/master_pipeline.py", "--source-path", str(source), "--skip-pull"]
result = subprocess.run(cmd, cwd=REPO, env=os.environ.copy())
if result.returncode != 0:
    raise SystemExit(f"NBA pipeline failed with exit code {result.returncode}")
print("NBA production pipeline completed")

In [ ]:
# Delivery boundary audit: stage ONLY the NBA delivery directory. This is
# intentionally not a commit and not a push; publication remains a
# separate human-reviewed action.
staged = subprocess.run(
    ["git", "diff", "--name-only", "--", str(DELIVERY)],
    cwd=REPO, capture_output=True, text=True, check=True,
).stdout.splitlines()
outside = [p for p in staged if not p.replace('\\', '/').startswith('nba-backend/data_delivery/')]
if outside:
    raise RuntimeError(f"delivery scope violation: {outside}")
subprocess.run(["git", "add", "--", str(DELIVERY)], cwd=REPO, check=True)
cached = subprocess.run(
    ["git", "diff", "--cached", "--name-only"], cwd=REPO,
    capture_output=True, text=True, check=True,
).stdout.splitlines()
outside_cached = [p for p in cached if not p.replace('\\', '/').startswith('nba-backend/data_delivery/')]
if outside_cached:
    raise RuntimeError(f"staged scope violation: {outside_cached}")
print(f"staged {len(cached)} NBA delivery files; no commit or push performed")